In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from eval import load

FIGURE_DIR = Path("../outputs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["svg.fonttype"] = "none"
plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
LOAD_CONFIGS = {
    "humaneval": {"dataset_size": 164, "numvariants": 51},
}


def _ensure_legacy_columns(df):
    df = df.copy()
    if "passed" not in df.columns and "Correct" in df.columns:
        df["passed"] = df["Correct"].astype(int)
    if "Correct" not in df.columns and "passed" in df.columns:
        df["Correct"] = df["passed"].astype(int)
    return df


def load_figure_data(model_name, *, numvariants=None):
    config = dict(LOAD_CONFIGS["humaneval"])
    if numvariants is not None:
        config["numvariants"] = numvariants
    df_data = load.load_humaneval(model_name, variant_type="retok", **config)
    df_data_0 = load.load_humaneval(model_name, variant_type="temperature", **config)
    return _ensure_legacy_columns(df_data), _ensure_legacy_columns(df_data_0)


def calculate_pass_k(n_total, num_correct, k):
    incorrect = n_total - num_correct
    if incorrect < k:
        return 1.0
    return 1.0 - comb(incorrect, k) / comb(n_total, k)


In [ ]:
df_datas_humaneval = {}
df_data_0s_humaneval = {}

all_models = ['meta-llama/Llama-3.1-8B-Instruct',
              'allenai/OLMo-2-1124-7B-Instruct',
              'Qwen/Qwen3-8B', 'google/gemma-3-27b-it','EleutherAI/pythia-6.9b',]#
all_models = ['allenai/OLMo-2-1124-7B-Instruct']

In [ ]:
from transformers import AutoTokenizer

for model_name in all_models:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"{model_name}: vocab size = {tokenizer.vocab_size}")


In [ ]:

# all_models = ['allenai/Olmo-3-7B-Instruct','allenai/OLMo-2-1124-7B-Instruct']
for model_name in all_models:
    if model_name == 'google/gemma-3-27b-it':
        sampling_size = 11
    else:
        sampling_size = 51
    print(model_name)
    try:
        df_data, df_data_0 = load_figure_data(model_name, numvariants=sampling_size)
    except Exception as e:
        print(f'Error loading humaneval {model_name}: {e}')
        raise

    print(f'{model_name} : df_data shape: {df_data.shape}, df_data_0 shape: {df_data_0.shape}')


    task_id_to_index = {task_id: i for i, task_id in enumerate(sorted(df_data.task_id.unique(), key=lambda x: int(x.split('/')[1])))}
    df_data['prompti'] = df_data['task_id'].map(task_id_to_index)
    df_data_0['prompti'] = df_data_0['task_id'].map(task_id_to_index)

    df_datas_humaneval[model_name] = df_data
    df_data_0s_humaneval[model_name] = df_data_0
    # try:
    #     df_data, df_data_0 = util.load_gsm8k(model_name)
    # except Exception as e:
    #     print(f'Error loading gsm8k {model_name}: {e}')

In [ ]:
df_data_0s_humaneval['allenai/OLMo-2-1124-7B-Instruct'].groupby('task_id').passed.sum()

In [ ]:
df_datas_humaneval['allenai/OLMo-2-1124-7B-Instruct'].groupby('task_id').passed.sum()

In [ ]:
df = df_data_0s_humaneval['allenai/OLMo-2-1124-7B-Instruct']

for tid in ['HumanEval/101', 'HumanEval/125','HumanEval/69','HumanEval/72',]:
    print(tid)
    print(df[df.task_id == tid].prompt.values[0])

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(figsize=(6,5))


dataset = 'humaneval'

colors_models = {all_models[i]:plt.cm.tab10((i)/len(all_models)) for i in range(len(all_models))}


for model_name in all_models:
    label = model_name.split('-')[-1]
    if label == '7B':
        label = 'base'
    df_data = df_datas_humaneval[model_name]
    df_data_0 = df_data_0s_humaneval[model_name]

    n_total_tasks = df_data.prompti.nunique()


    A_pass = sum(df_data_0.groupby('prompti').Correct.any())/n_total_tasks

    A_retok = sum(df_data.groupby('prompti').Correct.any())/n_total_tasks

    # A_pass, A_retok = 1.0,0.955


    N = df_data.prompti.value_counts().unique()[0]

    # ks = np.arange(1,min(N, 150),2)
    # ks = np.arange(1,min(N,40),2) #150
    ks = np.arange(1,50,2) #150

    color = colors_models[model_name]



    Cs = df_data.groupby('prompti').Correct.sum().values
    passk = np.array([[calculate_pass_k(N,Cs[task_id], k) for k in ks] for task_id in range(n_total_tasks)])
    ax.errorbar(ks, y=np.mean(passk, axis=0), color=color, label=f'pass@retok {label}', capsize=4, linestyle='-', linewidth=2)

    # ps = df_data.groupby('prompti').Correct.mean().values
    # alpha, beta = get_alpha_beta(ps)
    # ax.plot(ks, [pass_func(k,A_retok,alpha,beta) for k in ks], label=rf'A:{A_retok:0.2f} $\alpha$:{alpha:0.2f} $\beta$:{beta:0.2f}', color='firebrick', linestyle='--')


    Cs = df_data_0.groupby('prompti').Correct.sum().values
    ps = df_data_0.groupby('prompti').Correct.mean().values
    N = df_data_0.prompti.value_counts().unique()[0]


    passk_canon = [[calculate_pass_k(N,Cs[task_id], k) for k in ks] for task_id in range(n_total_tasks)]

    ax.errorbar(ks, y=np.mean(passk_canon, axis=0), color=color, label=rf'pass@k ({label})', capsize=4, linestyle='--', linewidth=2)


    # alpha, beta = get_alpha_beta(ps)
    # ax.plot(ks, [pass_func(k,A_pass,alpha,beta) for k in ks], label=rf'A:{A_pass:0.2f} $\alpha$:{alpha:0.2f} $\beta$:{beta:0.2f}', color='royalblue', linestyle='--')

labels = {model_name:model_name.split('/')[-1] for model_name in all_models}

leg1 = ax.legend(handles=[ax.plot([],[], color=colors_models[ds], label=labels[ds], linewidth=3)[0] for ds in all_models], bbox_to_anchor=(1,0.5), fontsize=12)


ax.grid('on')
ax.set_xlabel('k', fontsize=12)
ax.set_ylabel('Pass rate', fontsize=12)



labels2 = {'pass@retok':'-', 'pass@k':'--',}
# # #
# # #
leg2 = ax.legend(handles=[ax.plot([],[], color='k', linestyle=labels2[ds], label=ds)[0] for ds in labels2.keys()], bbox_to_anchor=(1.,1), fontsize=12, )
# #
ax.add_artist(leg1)#
# ax.set_yscale('log')

#change xtick fontsize
#change x tick fontsize
ax.tick_params(axis='both', which='major', labelsize=12)


fig.savefig(FIGURE_DIR / "4_humaneval_allmodels.svg", bbox_inches="tight")
# ax.set_title(f'{dataset} | Tasks : {n_total_tasks} | Replicates per task : {N}', fontsize=10)

# fig.suptitle('OLMo-2-1124-7B')

# fig.savefig('../outputs/figures/humaneval_allmodels.pdf', bbox_inches='tight')
